In [28]:
import pandas as pd
data_dir = "C:\\Users\\AM000098\\Desktop\\Lenovo\\ITE\\AllSim\\src\\myallsim\\data\\my_destination"
data_dir = "my_acceptance_data"
data = pd.read_csv(f"{data_dir}/acceptance_data_synth.csv")
train = pd.read_csv(f"{data_dir}/acceptance_data_synth_train.csv")
test = pd.read_csv(f"{data_dir}/acceptance_data_synth_test.csv")
train = train.drop(["Unnamed: 0"], axis = 1)
test = test.drop(["Unnamed: 0"], axis = 1)

In [2]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, log_loss, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import roc_auc_score

In [3]:
X_test = test.drop(columns=["ACC", "DEC_TIME"])
y_test = test[["ACC", "DEC_TIME"]]

train, val = train_test_split(train, test_size=0.05)

X_train = train.drop(columns=["ACC", "DEC_TIME"])
y_train = train[["ACC", "DEC_TIME"]]

X_val = val.drop(columns=["ACC", "DEC_TIME"])
y_val = val[["ACC", "DEC_TIME"]]

### Logistic Regression + Linear Regression

In [24]:
import numpy as np

In [22]:
# Logistic Regression for ACC prediction
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train["ACC"])

# Predict on the test set
y_pred_acc = log_reg.predict(X_test)

# Accuracy and Binary Cross-Entropy (Log Loss) for ACC
acc_log_reg = accuracy_score(y_test["ACC"], y_pred_acc)
bce_log_reg = log_loss(y_test["ACC"], log_reg.predict_proba(X_test))
auc_roc_acc = roc_auc_score(y_test["ACC"], log_reg.predict_proba(X_test)[:, 1])
print(auc_roc_acc)
print(f"Logistic Regression - ACC Accuracy: {acc_log_reg:.4f}, Binary Cross-Entropy: {bce_log_reg:.4f}")

0.7778867102396514
Logistic Regression - ACC Accuracy: 0.9652, Binary Cross-Entropy: 0.1241


C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
# Logistic Regression for ACC prediction
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train["ACC"])

# Predict on the test set
y_pred_acc = log_reg.predict(X_test)

# Get predicted probabilities for the positive class (class 1)
y_pred_proba_acc = log_reg.predict_proba(X_test)[:, 1]  # Probabilities for class 1

# Accuracy and Binary Cross-Entropy (Log Loss) for ACC
acc_log_reg = accuracy_score(y_test["ACC"], y_pred_acc)
bce_log_reg = log_loss(y_test["ACC"], y_pred_proba_acc)
auc_roc_acc = roc_auc_score(y_test["ACC"], y_pred_proba_acc)

# Manually calculate the per-sample binary cross-entropy
epsilon = 1e-15  # Small constant to avoid log(0)
y_pred_proba_acc_clipped = np.clip(y_pred_proba_acc, epsilon, 1 - epsilon)  # Clip probabilities to avoid log(0)
bce_loss_per_sample = - (y_test["ACC"] * np.log(y_pred_proba_acc_clipped) + 
                         (1 - y_test["ACC"]) * np.log(1 - y_pred_proba_acc_clipped))

# Calculate the mean and standard deviation of the per-sample BCE
bce_loss_mean = np.mean(bce_loss_per_sample)
bce_loss_std = np.std(bce_loss_per_sample)

# Print the results
print(f"AUC-ROC for ACC on Test Set: {auc_roc_acc:.4f}")
print(f"Logistic Regression - ACC Accuracy: {acc_log_reg:.4f}, Binary Cross-Entropy (Mean): {bce_log_reg:.4f}")
print(f"Binary Cross-Entropy (Mean): {bce_loss_mean:.4f}, Standard Deviation of BCE: {bce_loss_std:.4f}")

AUC-ROC for ACC on Test Set: 0.7779
Logistic Regression - ACC Accuracy: 0.9652, Binary Cross-Entropy (Mean): 0.1241
Binary Cross-Entropy (Mean): 0.1241, Standard Deviation of BCE: 0.5065


C:\Users\AM000098\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [31]:
from sklearn.metrics import brier_score_loss
# Calculate the per-sample Brier score (squared difference between predicted probabilities and actual outcomes)
brier_score_per_sample = (y_pred_proba_acc - y_test["ACC"])**2

# Calculate the mean and standard deviation of the per-sample Brier scores
brier_score_mean = np.mean(brier_score_per_sample)
brier_score_std = np.std(brier_score_per_sample)

# Print the mean and standard deviation of the Brier score
print(f"Brier Score (Mean) for ACC: {brier_score_mean:.4f}")
print(f"Brier Score (Standard Deviation) for ACC: {brier_score_std:.4f}")

Brier Score (Mean) for ACC: 0.0299
Brier Score (Standard Deviation) for ACC: 0.1445


In [26]:
# Logistic Regression for ACC prediction
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train["DEC_TIME"])

# Predict on the test set
y_pred_dec = lin_reg.predict(X_test)

# Accuracy and Binary Cross-Entropy (Log Loss) for ACC
mse_lin_reg = mean_squared_error(y_test["DEC_TIME"], y_pred_dec)


print(f"Linear Regression - MSE: {mse_lin_reg:.4f}")

Linear Regression - MSE: 0.1760


In [27]:
# Linear Regression for DEC_TIME prediction
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train["DEC_TIME"])

# Predict on the test set
y_pred_dec = lin_reg.predict(X_test)

# Calculate the Mean Squared Error (MSE)
mse_lin_reg = mean_squared_error(y_test["DEC_TIME"], y_pred_dec)

# Manually calculate the per-sample squared errors
squared_errors_per_sample = (y_test["DEC_TIME"] - y_pred_dec) ** 2

# Calculate the mean and standard deviation of the squared errors
mse_mean = np.mean(squared_errors_per_sample)
mse_std = np.std(squared_errors_per_sample)

# Print the results
print(f"Linear Regression - MSE: {mse_mean:.4f}, Standard Deviation of MSE: {mse_std:.4f}")

Linear Regression - MSE: 0.1760, Standard Deviation of MSE: 0.4864
